# SVAMITVA Staged Training Notebook (Single Universal `best.pt`)

This notebook trains the model in **separate stages/cells** (per head/layer group), while maintaining exactly one canonical checkpoint:

- `checkpoints/best.pt` (global universal best)

Each stage writes its own checkpoints under `checkpoints/staged/<stage_name>/`, then the stage best is promoted to the universal best only if its validation metric is better.

In [ ]:
# Cell 1: Environment and repo bootstrap
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "train.py").exists() and (p / "models").is_dir():
            return p
    raise RuntimeError("Could not find repository root (missing train.py/models).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")
print(f"Python: {sys.version.split()[0]}")

In [ ]:
# Cell 2: Imports
import copy
import json
import shutil
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

import torch

from data.dataset import create_dataloaders
from inference.predict import load_ensemble_pipeline
from models.losses import MultiTaskLoss
from models.model import EnsembleSvamitvaModel
from training.config import TrainingConfig
from training.trainer import Trainer

In [ ]:
# Cell 3: Global configuration (edit these)
TRAIN_DIRS = ["data/MAP1"]          # Add MAP directories here
VAL_DIR = "data/MAP1"               # Use same map for self-validation, or set None for auto-split

SAM2_CHECKPOINT = "checkpoints/sam2.1_hiera_base_plus.pt"
SAM2_MODEL_CFG = "configs/sam2.1/sam2.1_hiera_b+.yaml"

BATCH_SIZE = 1
NUM_WORKERS = 0
TILE_SIZE = 1024
TILE_OVERLAP = 192
VAL_SPLIT = 0.2
SPLIT_MODE = "map"
SEED = 42
FORCE_CPU = False

MAX_TRAIN_TILES = None   # Example: 24 for quick smoke run
MAX_VAL_TILES = None     # Example: 12 for quick smoke run

CHECKPOINT_ROOT = Path("checkpoints")
STAGED_ROOT = CHECKPOINT_ROOT / "staged"
UNIVERSAL_BEST_PATH = CHECKPOINT_ROOT / "best.pt"
UNIVERSAL_META_PATH = CHECKPOINT_ROOT / "universal_best_meta.json"

STAGED_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print("Universal best:", UNIVERSAL_BEST_PATH)

In [ ]:
# Cell 4: Build dataloaders once (reused across all stages)
train_loader, val_loader = create_dataloaders(
    train_dirs=[Path(p) for p in TRAIN_DIRS],
    val_dir=Path(VAL_DIR) if VAL_DIR else None,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    image_size=TILE_SIZE,
    tile_overlap=TILE_OVERLAP,
    val_split=VAL_SPLIT,
    split_mode=SPLIT_MODE,
    seed=SEED,
    max_train_tiles=MAX_TRAIN_TILES,
    max_val_tiles=MAX_VAL_TILES,
)
print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))

In [ ]:
# Cell 5: Build model + optional resume from universal best
DEVICE = torch.device(
    "cpu"
    if FORCE_CPU
    else ("cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"))
)
print("Device:", DEVICE)

model = EnsembleSvamitvaModel(
    num_roof_classes=5,
    pretrained=True,
    checkpoint_path=SAM2_CHECKPOINT,
    model_cfg=SAM2_MODEL_CFG,
    dropout=0.1,
)


def _extract_state_dict(ckpt_obj):
    if isinstance(ckpt_obj, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if isinstance(ckpt_obj.get(key), dict):
                return ckpt_obj[key]
        return ckpt_obj
    return ckpt_obj


def load_checkpoint_if_exists(model_obj, ckpt_path: Path, strict: bool = False) -> bool:
    if not ckpt_path.exists():
        print(f"No checkpoint at {ckpt_path}; starting from current model weights.")
        return False
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = _extract_state_dict(ckpt)
    incompatible = model_obj.load_state_dict(state, strict=strict)
    print(
        f"Loaded {ckpt_path} | missing={len(incompatible.missing_keys)} "
        f"unexpected={len(incompatible.unexpected_keys)}"
    )
    return True


load_checkpoint_if_exists(model, UNIVERSAL_BEST_PATH, strict=False)

In [ ]:
# Cell 6: Stage helpers (freeze/unfreeze, loss weights, universal best promotion)
ALL_BINARY_TASKS = [
    "building",
    "road",
    "road_centerline",
    "waterbody",
    "waterbody_line",
    "waterbody_point",
    "utility_line",
    "utility_point",
    "bridge",
    "railway",
]

ALL_HEAD_KEYS = [
    "building",
    "road",
    "road_centerline",
    "waterbody",
    "waterbody_line",
    "waterbody_point",
    "utility_line",
    "utility_point",
    "bridge",
    "railway",
]

STAGE_RESULTS: List[Dict[str, object]] = []


def set_trainable_layers(
    model_obj: EnsembleSvamitvaModel,
    train_head_keys: List[str],
    train_decoder: bool,
    train_backbone: bool,
) -> None:
    for p in model_obj.parameters():
        p.requires_grad = False

    if train_backbone:
        for p in model_obj.encoder.parameters():
            p.requires_grad = True

    if train_decoder:
        for p in model_obj.decoder.parameters():
            p.requires_grad = True

    for head_key in train_head_keys:
        if head_key not in model_obj.heads:
            raise KeyError(f"Unknown head key: {head_key}")
        for p in model_obj.heads[head_key].parameters():
            p.requires_grad = True



def trainable_param_count(model_obj: torch.nn.Module) -> int:
    return sum(p.numel() for p in model_obj.parameters() if p.requires_grad)


def make_loss_weights(active_binary_tasks: List[str], include_roof: bool) -> Dict[str, float]:
    weights = {task: 0.0 for task in ALL_BINARY_TASKS}
    for task in active_binary_tasks:
        if task not in weights:
            raise KeyError(f"Unknown binary task: {task}")
        weights[task] = 1.0
    weights["roof_type"] = 0.5 if include_roof else 0.0
    return weights


def current_universal_score() -> float:
    if not UNIVERSAL_META_PATH.exists():
        return float("-inf")
    with open(UNIVERSAL_META_PATH, "r", encoding="utf-8") as f:
        meta = json.load(f)
    return float(meta.get("best_score", float("-inf")))


def update_universal_best(stage_name: str, stage_best_path: Path, stage_score: float) -> bool:
    prev = current_universal_score()
    promote = (not UNIVERSAL_BEST_PATH.exists()) or (stage_score >= prev)

    if promote:
        shutil.copy2(stage_best_path, UNIVERSAL_BEST_PATH)
        meta = {
            "best_score": float(stage_score),
            "metric": "avg_iou",
            "source_stage": stage_name,
            "source_checkpoint": str(stage_best_path),
            "updated_at": datetime.utcnow().isoformat() + "Z",
        }
        with open(UNIVERSAL_META_PATH, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)
        print(f"[PROMOTED] {stage_name} -> universal best (avg_iou={stage_score:.6f})")
    else:
        print(
            f"[KEPT] Universal best unchanged (current={prev:.6f}, "
            f"stage={stage_score:.6f}, stage={stage_name})"
        )
    return promote


def run_stage(
    stage_name: str,
    train_head_keys: List[str],
    active_binary_tasks: List[str],
    include_roof: bool,
    epochs: int,
    learning_rate: float,
    train_decoder: bool,
    train_backbone: bool,
    dropout: float = 0.1,
) -> Dict[str, object]:
    stage_dir = STAGED_ROOT / stage_name
    stage_logs = stage_dir / "logs"
    stage_dir.mkdir(parents=True, exist_ok=True)
    stage_logs.mkdir(parents=True, exist_ok=True)

    # Load universal checkpoint before every stage (stable stage entry point)
    load_checkpoint_if_exists(model, UNIVERSAL_BEST_PATH, strict=False)

    set_trainable_layers(
        model_obj=model,
        train_head_keys=train_head_keys,
        train_decoder=train_decoder,
        train_backbone=train_backbone,
    )

    freeze_backbone_epochs = 0 if train_backbone else max(1, int(epochs))

    cfg = TrainingConfig(
        train_dirs=[Path(p) for p in TRAIN_DIRS],
        val_dir=Path(VAL_DIR) if VAL_DIR else None,
        checkpoint_dir=stage_dir,
        log_dir=stage_logs,
        batch_size=BATCH_SIZE,
        num_epochs=int(epochs),
        learning_rate=float(learning_rate),
        tile_size=TILE_SIZE,
        tile_overlap=TILE_OVERLAP,
        split_mode=SPLIT_MODE,
        val_split=VAL_SPLIT,
        num_workers=NUM_WORKERS,
        seed=SEED,
        freeze_backbone_epochs=freeze_backbone_epochs,
        force_cpu=FORCE_CPU,
        experiment_name=stage_name,
        dropout=float(dropout),
        sam2_checkpoint=SAM2_CHECKPOINT,
        sam2_model_cfg=SAM2_MODEL_CFG,
    )
    cfg.loss_weights = make_loss_weights(active_binary_tasks, include_roof)

    loss_fn = MultiTaskLoss(weights=cfg.loss_weights, num_roof_classes=cfg.num_roof_classes)
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_fn=loss_fn,
        config=cfg,
        start_epoch=1,
    )

    print("=" * 80)
    print(f"Stage: {stage_name}")
    print(f"Train heads: {train_head_keys}")
    print(f"Active binary tasks: {active_binary_tasks} | include_roof={include_roof}")
    print(f"Train decoder={train_decoder} | train backbone={train_backbone}")
    print(f"Trainable params: {trainable_param_count(model):,}")

    trainer.fit()

    stage_best = stage_dir / "best.pt"
    if not stage_best.exists():
        raise FileNotFoundError(f"Stage best checkpoint not found: {stage_best}")

    stage_score = float(trainer.ckpt_mgr.best_score)
    promoted = update_universal_best(stage_name, stage_best, stage_score)

    # Load stage best into working model for subsequent stages.
    load_checkpoint_if_exists(model, stage_best, strict=False)

    result = {
        "stage": stage_name,
        "best_score": stage_score,
        "best_epoch": int(trainer.ckpt_mgr.best_epoch),
        "stage_best": str(stage_best),
        "promoted_to_universal": bool(promoted),
    }
    STAGE_RESULTS.append(result)
    return result

## Stage Cells
Run each stage cell independently. Every stage trains only the selected heads/layers, then tries to promote its best checkpoint to `checkpoints/best.pt`.

In [ ]:
# Cell 7: Stage 1 - Building + Roof (building head only)
stage_1 = run_stage(
    stage_name="stage01_building_roof",
    train_head_keys=["building"],
    active_binary_tasks=["building"],
    include_roof=True,
    epochs=3,
    learning_rate=3e-4,
    train_decoder=False,
    train_backbone=False,
)
stage_1

In [ ]:
# Cell 8: Stage 2 - Roads (road + centerline heads)
stage_2 = run_stage(
    stage_name="stage02_roads",
    train_head_keys=["road", "road_centerline"],
    active_binary_tasks=["road", "road_centerline"],
    include_roof=False,
    epochs=3,
    learning_rate=3e-4,
    train_decoder=False,
    train_backbone=False,
)
stage_2

In [ ]:
# Cell 9: Stage 3 - Water features
stage_3 = run_stage(
    stage_name="stage03_water",
    train_head_keys=["waterbody", "waterbody_line", "waterbody_point"],
    active_binary_tasks=["waterbody", "waterbody_line", "waterbody_point"],
    include_roof=False,
    epochs=3,
    learning_rate=3e-4,
    train_decoder=False,
    train_backbone=False,
)
stage_3

In [ ]:
# Cell 10: Stage 4 - Utility features
stage_4 = run_stage(
    stage_name="stage04_utility",
    train_head_keys=["utility_line", "utility_point"],
    active_binary_tasks=["utility_line", "utility_point"],
    include_roof=False,
    epochs=3,
    learning_rate=3e-4,
    train_decoder=False,
    train_backbone=False,
)
stage_4

In [ ]:
# Cell 11: Stage 5 - Bridge + Railway
stage_5 = run_stage(
    stage_name="stage05_infra",
    train_head_keys=["bridge", "railway"],
    active_binary_tasks=["bridge", "railway"],
    include_roof=False,
    epochs=3,
    learning_rate=3e-4,
    train_decoder=False,
    train_backbone=False,
)
stage_5

In [ ]:
# Cell 12: Stage 6 - Joint full fine-tuning (all heads + backbone)
stage_6 = run_stage(
    stage_name="stage06_joint_finetune",
    train_head_keys=ALL_HEAD_KEYS,
    active_binary_tasks=ALL_BINARY_TASKS,
    include_roof=True,
    epochs=3,
    learning_rate=1e-4,
    train_decoder=True,
    train_backbone=True,
)
stage_6

In [ ]:
# Cell 13: Stage summary + universal best metadata
try:
    import pandas as pd
    display(pd.DataFrame(STAGE_RESULTS))
except Exception:
    print(STAGE_RESULTS)

if UNIVERSAL_META_PATH.exists():
    print("\nUniversal best metadata:")
    print(UNIVERSAL_META_PATH.read_text(encoding="utf-8"))
print("\nUniversal best path:", UNIVERSAL_BEST_PATH)

In [ ]:
# Cell 14: Validate universal best on first available map tif

def find_first_tif(dir_path: Path) -> Optional[Path]:
    if not dir_path.exists():
        return None
    for ext in ("*.tif", "*.tiff", "*.TIF", "*.TIFF"):
        hits = sorted(dir_path.glob(ext))
        if hits:
            return hits[0]
    return None

val_map_dir = Path(VAL_DIR) if VAL_DIR else Path(TRAIN_DIRS[0])
val_tif = find_first_tif(val_map_dir)
if val_tif is None:
    raise FileNotFoundError(f"No tif found in {val_map_dir}")

predictor = load_ensemble_pipeline(
    weights_path=str(UNIVERSAL_BEST_PATH),
    yolo_path="checkpoints/yolov8s.pt",
    device=DEVICE,
    use_tta=False,
    yolo_conf=0.25,
    yolo_iou=0.45,
    tile_size=TILE_SIZE,
    overlap=TILE_OVERLAP,
)

preds = predictor.predict_tif(val_tif)
pixel_counts = {k: int(v.sum()) for k, v in preds.items() if k.endswith("_mask")}
print("Validation tif:", val_tif)
print("Non-zero pixel counts:")
for k in sorted(pixel_counts.keys()):
    print(f"  {k}: {pixel_counts[k]}")

## Notes

- This notebook keeps **one universal model file** at `checkpoints/best.pt`.
- Per-stage checkpoints remain under `checkpoints/staged/` for traceability.
- Promotion policy uses `avg_iou` (higher is better).
- If you only want one final artifact, archive/remove `checkpoints/staged/` after training.